In [1]:
import os
import re
import hashlib
from datasets import load_dataset, Dataset

# ---------------------------------------------------------------------------
# 1) DROP ENTIRE FILE if it matches these (generated/vendored code, not
#    real hand-written logic worth learning from)
# ---------------------------------------------------------------------------
DROP_FILE_PATTERNS = re.compile(
    r"(created automatically by swig|don't modify this file|"
    r"do not edit|do not modify|autogenerated|auto-generated|"
    r"generated by the protocol buffer compiler|"
    r"generated from .*\.txt|generated by .*gencodec|"
    r"this file was generated|"
    r"generated by cython|# generated by |"
    r"swig_setattr|swig_getattr)",
    re.IGNORECASE,
)

# ---------------------------------------------------------------------------
# 2) STRIP long license header blocks (top-of-file boilerplate)
# ---------------------------------------------------------------------------
def strip_license_header(content):
    lines = content.split("\n")
    end = 0
    saw_license_kw = False
    for i, line in enumerate(lines[:80]):
        stripped = line.strip()
        if stripped.startswith("#"):
            if re.search(r"copyright|license", stripped, re.IGNORECASE):
                saw_license_kw = True
            end = i + 1
        elif stripped == "":
            end = i + 1
        else:
            break
    if saw_license_kw:
        return "\n".join(lines[end:]).lstrip("\n")
    return content


# ---------------------------------------------------------------------------
# 3) STRIP huge non-Python metadata blocks (Ansible DOCUMENTATION / EXAMPLES
#    / RETURN triple-quoted blobs) -- these are not Python logic.
# ---------------------------------------------------------------------------
METADATA_BLOCK_PATTERN = re.compile(
    r"^(?:DOCUMENTATION|EXAMPLES|RETURN|ANSIBLE_METADATA)\s*=\s*"
    r"('''|\"\"\")(?:.*?)\1\s*\n?",
    re.DOTALL | re.MULTILINE,
)


def strip_metadata_blocks(content):
    return METADATA_BLOCK_PATTERN.sub("", content)


# ---------------------------------------------------------------------------
# 4) REDACT email addresses anywhere in the file
# ---------------------------------------------------------------------------
EMAIL_PATTERN = re.compile(r"[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+")


def redact_emails(content):
    return EMAIL_PATTERN.sub("<EMAIL>", content)


# ---------------------------------------------------------------------------
# 5) STRIP low-value boilerplate comment lines (Filename:, Author:, etc.)
#    Keep real docstrings untouched -- this only targets '#'-style lines.
# ---------------------------------------------------------------------------
BOILERPLATE_LINE_PATTERN = re.compile(
    r"^[ \t]*#[ \t]*(filename|author|created|version|date|maintainer)[ \t]*:.*\n?",
    re.IGNORECASE | re.MULTILINE,
)


def strip_boilerplate_lines(content):
    return BOILERPLATE_LINE_PATTERN.sub("", content)


# ---------------------------------------------------------------------------
# Pipeline
# ---------------------------------------------------------------------------
def clean(content):
    content = strip_license_header(content)
    content = strip_metadata_blocks(content)
    content = strip_boilerplate_lines(content)
    content = redact_emails(content)
    return content.strip()


def should_drop_file(content):
    if DROP_FILE_PATTERNS.search(content[:1000]):
        return True
    if len(content) < 100 or len(content) > 20000:
        return True
    return False


# ---------------------------------------------------------------------------
# Config -- adjust these
# ---------------------------------------------------------------------------
TARGET_COUNT = 500_000                          # how many cleaned files to collect
HF_REPO_ID = "your-username/codeparrot-clean-py-filtered"  # <-- change this
PRIVATE_REPO = True                             # keep True -- see licensing note below

# ---------------------------------------------------------------------------
# Run: stream, clean, dedupe, collect
# ---------------------------------------------------------------------------
ds = load_dataset("codeparrot/codeparrot-clean", split="train", streaming=True)

seen_hashes = set()
cleaned_examples = []

for example in ds:
    if example["autogenerated"]:
        continue

    raw = example["content"]
    if should_drop_file(raw):
        continue

    cleaned = clean(raw)
    if not cleaned:
        continue

    h = hashlib.sha256(cleaned.encode("utf-8", errors="ignore")).hexdigest()
    if h in seen_hashes:
        continue
    seen_hashes.add(h)

    cleaned_examples.append({"content": cleaned})

    if len(cleaned_examples) % 10_000 == 0:
        print(f"Collected {len(cleaned_examples)} cleaned files so far...")

    if len(cleaned_examples) >= TARGET_COUNT:
        break

print(f"Done cleaning. Total kept: {len(cleaned_examples)}")

# ---------------------------------------------------------------------------
# Push to Hugging Face Hub
# ---------------------------------------------------------------------------
# Requires being logged in first, e.g. in your terminal:
#   huggingface-cli login
# (paste your token there when prompted -- never hardcode it in this script)

final_ds = Dataset.from_list(cleaned_examples)


README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/54 [00:00<?, ?it/s]

Collected 10000 cleaned files so far...
Collected 20000 cleaned files so far...
Collected 30000 cleaned files so far...
Collected 40000 cleaned files so far...
Collected 50000 cleaned files so far...
Collected 60000 cleaned files so far...
Collected 70000 cleaned files so far...
Collected 80000 cleaned files so far...
Collected 90000 cleaned files so far...
Collected 100000 cleaned files so far...
Collected 110000 cleaned files so far...
Collected 120000 cleaned files so far...
Collected 130000 cleaned files so far...
Collected 140000 cleaned files so far...
Collected 150000 cleaned files so far...
Collected 160000 cleaned files so far...
Collected 170000 cleaned files so far...
Collected 180000 cleaned files so far...
Collected 190000 cleaned files so far...
Collected 200000 cleaned files so far...
Collected 210000 cleaned files so far...
Collected 220000 cleaned files so far...
Collected 230000 cleaned files so far...
Collected 240000 cleaned files so far...
Collected 250000 cleaned 

In [ ]:
final_ds.push_to_hub(
    "Ananda100/python-clean",
    token="YOUR_HF_TOKEN"
)

Uploading the dataset shards:   0%|          | 0/6 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/5 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Creating parquet from Arrow format:   0%|          | 0/5 [00:00<?, ?ba/s]

...

New Data Upload: |          |  0.00B /  0.00B            

Creating parquet from Arrow format:   0%|          | 0/5 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

CommitInfo(commit_url='https://huggingface.co/datasets/Ananda100/python-clean/commit/cde3683bfe974aec5e4243ed03e481982ae4612f', commit_message='Upload dataset', commit_description='', oid='cde3683bfe974aec5e4243ed03e481982ae4612f', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/Ananda100/python-clean', endpoint='https://huggingface.co', repo_type='dataset', repo_id='Ananda100/python-clean'), pr_revision=None, pr_num=None)

In [4]:
from datasets import load_dataset

ds = load_dataset(
    "Ananda100/python-clean",
    split="train",
    streaming=True  # set to False if you want to load it fully into memory
)

for i, example in enumerate(ds):
    print(f"--- Example {i+1} ---")
    print(example["content"][:800])
    print()
    if i == 9:
        break

README.md:   0%|          | 0.00/288 [00:00<?, ?B/s]

--- Example 1 ---
from autobahn.asyncio.websocket import WebSocketClientProtocol, \
                                       WebSocketClientFactory

import asyncio



class MyClientProtocol(WebSocketClientProtocol):

   def onConnect(self, response):
      print("Server connected: {0}".format(response.peer))

   @asyncio.coroutine
   def onOpen(self):
      print("WebSocket connection open.")

      ## start sending messages every second ..
      while True:
         self.sendMessage(u"Hello, world!".encode('utf8'))
         self.sendMessage(b"\x00\x01\x03\x04", isBinary = True)
         yield from asyncio.sleep(1)

   def onMessage(self, payload, isBinary):
      if isBinary:
         print("Binary message received: {0} bytes".format(len(payload)))
      else:
         print("Text message received: {0}".form

--- Example 2 ---
from itertools import chain

from django.utils.itercompat import is_iterable


class Tags:
    """
    Built-in tags for internal checks.
    """
    admin = 'adm